# S3.3b — Verifier-B (BanglaBERT recipe, **retrained on R2**)

**Two code cells, split for the same reason as the S3.2 notebook:** a single
cell cannot be half-run, and checking preflight output by interrupting the real
run still spends GPU quota.

- **Cell 1 — preflight.** Clone, install, attach data, run the wall tests, run
  the dry run. Minutes, no GPU work, safe interactively.
- **Cell 2 — the real run.** 5 fine-tuning runs (5 seeds x **1** learning rate).
  Roughly 35–45 minutes on a T4, i.e. about a tenth of S3.2.

## 🔴 Read this before running

**Verifier-B is not an S3.2 checkpoint.** Decision 16 called it *"the fine-tuned
BanglaBERT from S3.2"*, but `configs/s3_backbone.yaml` sets `role: A`, so every
S3.2 arm trained on **R1 — Verifier-A's data**. Taken literally that sentence
would have put both verifiers on the same rows and voided inviolable rule 6,
which is the wall RQ5 measures. What runs here is the **recipe** — same
backbone, same budget, same seeds — retrained on **R2's 888 rows**. No
checkpoint is loaded, and `tests/test_s3_verifiers.py` fails if the config says
`role: A`.

**One learning rate, not two.** `schneider2025overtuning` find ~10% of tuned HPO
runs generalise *worse* than the default, worst under exactly this run's
conditions — small data, holdout rather than CV, binary task, accuracy-type
metric. lr is taken from pipeline §3.1 and never selected. Sabbir's call,
2026-08-11; see `protocol.md` §S3.3 decision 1.

**The persisted artifact is the seed-42 model**, declared before any score
exists — *not* the best of five. The other four seeds are the sensitivity band.

## How to run

1. **Accelerator → GPU T4**, **Internet → On**, attach `bn_clean.csv` via
   *+ Add Input*.
2. Use **Save & Run All (Commit)**, not an interactive session. `/kaggle/working`
   does not survive across sessions, and that cost four GPU-hours on 2026-08-09.
3. Download `s3d_verifier_b.zip` from the output and commit the `results/` files
   **together with the lab-notebook entry** — a commit that touches `results/`
   without touching `docs/lab_notebook.md` is a defect (CLAUDE.md).

⚠️ **Read `protocol.md` §"S3.3 pre-commitment" before you read the number.** In
particular: *no claim that either verifier is better than the other may be made
from dev-82.* One item is 0.0122 macro-F1 and the expected A−B gap is under two
reviews.


## Cell 1 — preflight (minutes, no GPU)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  S3.3b preflight. Everything here is cheap and reversible.
#
#  This notebook is a RUNNER. It clones, installs, checks, and calls the
#  script. No logic lives here: anything computed in a notebook cell cannot
#  enter the paper (CLAUDE.md, working conventions).
# ══════════════════════════════════════════════════════════════════════════

%cd /kaggle/working
!rm -rf /kaggle/working/thesis
!git clone --depth 1 https://github.com/alphapie77/BSc_Thesis.git /kaggle/working/thesis
%cd /kaggle/working/thesis

# ⚠️ CHECK THIS. If the commit predates the S3.3 pre-registration, the clone
#    does not contain the protocol section this run is supposed to follow.
!git log --oneline -1

# transformers is pinned below 5 for the same reason as S3.2: Kaggle ships
# 5.0.0, and the whole point of pinning is that Verifier-B must be the S3.2
# BanglaBERT RECIPE. Coakley et al. (2022) measured >6 pp of accuracy variation
# from environment alone, so "same recipe, different transformers major
# version" is not the same recipe.
!pip install -q 'transformers<5' datasets pyyaml scikit-learn joblib

import transformers
print('transformers', transformers.__version__)

import shutil
from pathlib import Path

hits = sorted(Path('/kaggle/input').rglob('bn_clean.csv'))
if not hits:
    visible = [str(p) for p in Path('/kaggle/input').rglob('*') if p.is_file()][:20]
    raise FileNotFoundError(
        "bn_clean.csv is not under /kaggle/input. Attach it with '+ Add Input'. "
        f"Visible now: {visible or 'NOTHING'}"
    )
Path('data/cleaned').mkdir(parents=True, exist_ok=True)
shutil.copy(hits[0], 'data/cleaned/bn_clean.csv')
print('input:', hits[0])

# ── Gate 1: the wall. If Verifier-B's training ids touch R1, or the config
# ── says role A, or the dev slice is not the shared 82, stop here — every
# ── number after that point would be contaminated in a way no result file
# ── would show.
!python -m pytest tests/test_s3_verifiers.py tests/test_s3_backbone.py tests/test_split_map.py -q

# ── Gate 2: the dry run. Proves the plumbing end to end and re-checks that n
# ── is still 888/82 on THIS host, before any weights are downloaded. Its
# ── predictions come from a random number generator and mean nothing, which
# ── is why it writes to results/_dryrun/ and cannot touch results/.
!python -m src.verifier.train_verifier_b --config configs/s3d_verifier_b.yaml --dry-run

# ── The environment snapshot is mandatory on any non-local host (fact (env)).
# ── S3.3b's numbers are attributable to THIS file, not to requirements.lock.txt.
!python -m src.common.env_snapshot --out results/env_snapshot_s3d_kaggle.json


## Cell 2 — the real run (~40 min, GPU)

Five fine-tuning runs. Do not run this interactively — reaching it via
**Save & Run All** is the point of the split.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  The real run: 5 seeds x 1 learning rate, BanglaBERT on R2's 888 rows.
#
#  The dry-run artifacts live in results/_dryrun/ and are NOT overwritten
#  here — they are in a different directory on purpose, so a stub's output can
#  never sit next to a real result under the same name.
# ══════════════════════════════════════════════════════════════════════════

%cd /kaggle/working/thesis

!python -m src.verifier.train_verifier_b --config configs/s3d_verifier_b.yaml

import json
from pathlib import Path

r = json.loads(Path('results/s3d_verifier_b.json').read_text(encoding='utf-8'))['result']
assert r['dry_run'] is False, 'this result came from a DRY RUN — stop'
assert r['role'] == 'B'
print(open('results/s3d_verifier_b.md', encoding='utf-8').read())

# Zip whatever exists, even on a partial run: five independent seeds mean a
# session killed at seed 4 still yields four usable rows, and losing them to
# a failed final write would be the same waste as 2026-08-09.
import zipfile
OUT = Path('/kaggle/working/s3d_verifier_b.zip')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('results/s3d_verifier_b.md',
              'results/s3d_verifier_b.json',
              'results/s3d_verifier_b_per_seed.csv',
              'results/s3d_verifier_b_dev_predictions.csv',
              'results/env_snapshot_s3d_kaggle.json'):
        if Path(f).exists():
            z.write(f)
    # The weights themselves: Phase 4's S6 scoring needs this artifact, and
    # re-training it later to get it back would not reproduce it bit for bit
    # across a changed environment.
    for p in Path('artifacts').rglob('*'):
        if p.is_file():
            z.write(p)
print('wrote', OUT)


## Cell 3 — Verifier-A (minutes, CPU is fine)

Only needed if Verifier-A was not fitted locally. It is a frozen LaBSE encode
plus a logistic regression — seconds of compute after the encoder downloads —
and it writes `results/s3c_verifier_a.*` plus `artifacts/verifier_a.joblib`.

The script **refuses to finish** unless dev macro-F1 reproduces S3.2b's measured
**0.9866** to within half a dev item. It is the same model on the same rows, so
a drift means the LaBSE revision, the split map or the K=2 assignments moved —
and that is a stop, not a new number.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  S3.3a -- Verifier-A. Self-contained: clones and installs only if that has
#  not already happened in this session, so it runs standalone or as cell 3.
# ══════════════════════════════════════════════════════════════════════════

from pathlib import Path
import shutil, subprocess, sys

REPO = Path('/kaggle/working/thesis')
if not REPO.exists():
    print('no clone in this session -- cloning')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/alphapie77/BSc_Thesis.git', str(REPO)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers<5',
                    'sentence-transformers', 'scikit-learn', 'pyyaml', 'joblib'], check=True)
else:
    print('reusing the clone from cell 1')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sentence-transformers', 'joblib'], check=True)

%cd /kaggle/working/thesis
!git log --oneline -1

hits = sorted(Path('/kaggle/input').rglob('bn_clean.csv'))
if not hits:
    raise FileNotFoundError("attach bn_clean.csv with '+ Add Input'")
Path('data/cleaned').mkdir(parents=True, exist_ok=True)
shutil.copy(hits[0], 'data/cleaned/bn_clean.csv')

!python -m pytest tests/test_s3_verifiers.py -q

!python -m src.verifier.train_verifier_a --config configs/s3c_verifier_a.yaml

print(open('results/s3c_verifier_a.md', encoding='utf-8').read())

import zipfile
OUT = Path('/kaggle/working/s3c_verifier_a.zip')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('results/s3c_verifier_a.md',
              'results/s3c_verifier_a.json',
              'results/s3c_verifier_a_dev_predictions.csv',
              'artifacts/verifier_a.joblib'):
        if Path(f).exists():
            z.write(f)
print('wrote', OUT)
